# Project 3 — Student Success Predictor

**Goal**: Predict whether a student will **Pass or Fail** based on study habits and academic performance.

| Phase | Task |
|---|---|
| 1 | Data Loading & Exploration (EDA) |
| 2 | Data Cleaning & Missing Value Check |
| 3 | Categorical Encoding |
| 4 | Feature Scaling + Train/Test Split |
| 5 | Model Training (Logistic Regression) |
| 6 | Model Evaluation |
| 7 | Visualisation |
| 8 | Predict a New Student |

**Dataset**: `student_small_dataset.csv` — 20 students, 6 columns

| Feature | Type | Description |
|---|---|---|
| StudyHours | Numeric | Hours studied per day |
| Attendance | Numeric | Attendance % |
| PastScore | Numeric | Previous exam score |
| Internet | Categorical | Has internet? Yes/No |
| SleepHours | Numeric | Hours slept per day |
| Passed | Target | Did student pass? Yes/No |

## Phase 1 — Data Loading & Exploration (EDA)

Always explore data before building a model:
- How many rows and columns?
- What data types exist?
- Are there missing values?
- What does the distribution look like?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── LOAD DATA ────────────────────────────────────────────────────────────────
df = pd.read_csv('student_small_dataset.csv')

print('=== First 5 Rows ===')
print(df.head())

print(f'\n=== Dataset Shape ===')
print(f'Rows: {df.shape[0]}  |  Columns: {df.shape[1]}')

print('\n=== Column Data Types ===')
print(df.dtypes)

# isnull().sum() counts NaN per column — important to check before anything else
print('\n=== Missing Values per Column ===')
print(df.isnull().sum())

# describe(include='all') gives stats for both numeric and categorical columns
print('\n=== Summary Statistics ===')
print(df.describe(include='all').round(2))

# Value counts of target — check if classes are balanced
print('\n=== Target Distribution (Passed) ===')
print(df['Passed'].value_counts())
print(f'Class balance: {(df["Passed"]=="Yes").mean()*100:.0f}% Pass  |  {(df["Passed"]=="No").mean()*100:.0f}% Fail')

## Phase 2 — Data Cleaning

Check for:
- **Missing values** — impute or drop
- **Duplicate rows** — remove
- **Outliers** — check numeric column ranges

In [ ]:
# ── DUPLICATES ────────────────────────────────────────────────────────────────
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')
if duplicates > 0:
    df = df.drop_duplicates()
    print(f'  Removed {duplicates} duplicate(s)')

# ── MISSING VALUES ────────────────────────────────────────────────────────────
missing = df.isnull().sum()
if missing.sum() == 0:
    print('No missing values found — dataset is clean')
else:
    print('Missing values found:')
    print(missing[missing > 0])

# ── OUTLIER CHECK (IQR method) ────────────────────────────────────────────────
# IQR = Q3 - Q1
# Anything beyond Q1 - 1.5*IQR or Q3 + 1.5*IQR is a potential outlier
print('\n=== Outlier Check (numeric columns) ===')
numeric_cols = df.select_dtypes(include='number').columns.tolist()
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    status = f'{len(outliers)} outlier(s) found' if len(outliers) > 0 else 'clean'
    print(f'  {col:<15}: range [{df[col].min()}, {df[col].max()}]  bounds [{lower:.1f}, {upper:.1f}]  → {status}')

print(f'\nFinal dataset: {df.shape[0]} rows  (ready for modelling)')

## Phase 3 — Categorical Encoding

ML models need numbers — text columns must be converted.

- `Internet` (Yes/No) → **LabelEncoder** → 0 or 1
- `Passed` (Yes/No) → **LabelEncoder** → 0 (Fail) or 1 (Pass)

> Note: we use the same `LabelEncoder` instance for `Passed` since Yes→1 and No→0 (alphabetical order: No < Yes).

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()  # always work on a copy to preserve the original

le = LabelEncoder()

# Encode Internet: No=0, Yes=1 (alphabetical order)
df_encoded['Internet'] = le.fit_transform(df_encoded['Internet'])
print(f'Internet encoding: No→{le.transform(["No"])[0]}, Yes→{le.transform(["Yes"])[0]}')

# Encode Passed: No=0, Yes=1 → this becomes our target (0=Fail, 1=Pass)
df_encoded['Passed'] = le.fit_transform(df_encoded['Passed'])
print(f'Passed encoding:   No→{0} (Fail), Yes→{1} (Pass)')

print('\n=== After Encoding ===')
print(df_encoded.head(10))

print('\n=== Data Types After Encoding ===')
print(df_encoded.dtypes)

## Phase 4 — Feature Scaling + Train / Test Split

**Logistic Regression is sensitive to feature scale** — StudyHours (1–10) and Attendance (40–100) have very different ranges. `StandardScaler` brings both to the same scale (mean=0, std=1).

**Key rule**: always split BEFORE scaling. Fit the scaler on training data only — never on test data.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Features used for prediction (Internet excluded — low signal for this small dataset)
features = ['StudyHours', 'Attendance', 'PastScore', 'SleepHours']

X = df_encoded[features]  # input features
y = df_encoded['Passed']  # target: 0=Fail, 1=Pass

print(f'Features : {features}')
print(f'Target   : Passed  (0=Fail, 1=Pass)')
print(f'\nX shape: {X.shape}  |  y shape: {y.shape}')
print(f'Class distribution — Fail: {(y==0).sum()}, Pass: {(y==1).sum()}')

# Split: 80% train, 20% test
# random_state=42 makes the split reproducible every run
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'\nTrain samples: {len(X_train)}  |  Test samples: {len(X_test)}')

# Scale AFTER splitting — fit on train only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # learn mean+std from train, apply
X_test_scaled  = scaler.transform(X_test)        # apply same stats to test (no re-fitting)

print('\nScaling applied (fit on training data only)')
print(f'Learned means  : {dict(zip(features, scaler.mean_.round(2)))}')
print(f'Learned std devs: {dict(zip(features, scaler.scale_.round(2)))}')

## Phase 5 — Model Training (Logistic Regression)

**Logistic Regression** predicts the probability of belonging to a class (Pass or Fail).
- Uses a sigmoid curve to output a value between 0 and 1
- If probability > 0.5 → predict Pass (1), else → predict Fail (0)
- Learns one coefficient (weight) per feature — larger absolute weight = more influence

In [ ]:
from sklearn.linear_model import LogisticRegression

# max_iter: maximum optimisation steps (increase if ConvergenceWarning appears)
model = LogisticRegression(max_iter=1000, random_state=42)

# fit() learns the decision boundary from training data
model.fit(X_train_scaled, y_train)

print('=== What the Model Learned ===')
print(f'Intercept (base log-odds): {model.intercept_[0]:.4f}')

# coef_ shows how strongly each feature pushes toward Pass (positive) or Fail (negative)
coef_df = pd.DataFrame({'Feature': features, 'Coefficient': model.coef_[0]})
coef_df = coef_df.sort_values('Coefficient', ascending=False)

print('\nFeature Coefficients (after scaling):')
for _, row in coef_df.iterrows():
    direction = '→ pushes toward PASS' if row['Coefficient'] > 0 else '→ pushes toward FAIL'
    bar = '█' * int(abs(row['Coefficient']) * 5)
    print(f'  {row["Feature"]:<15} {row["Coefficient"]:>+7.3f}  {direction}  {bar}')

# Predictions
y_pred      = model.predict(X_test_scaled)           # 0 or 1 for each test student
y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]  # P(Pass) for each student

print('\n=== Test Set Predictions ===')
result_df = X_test.copy().reset_index(drop=True)
result_df['Actual']    = y_test.values
result_df['Predicted'] = y_pred
result_df['P(Pass)']   = y_pred_prob.round(3)
result_df['Correct']   = result_df['Actual'] == result_df['Predicted']
print(result_df.to_string(index=False))

## Phase 6 — Model Evaluation

For a classification problem we look at:
- **Accuracy** — overall correct predictions
- **Precision** — of all predicted Pass, how many really passed?
- **Recall** — of all actual Pass, how many did the model catch?
- **F1 Score** — balance of precision and recall
- **Confusion Matrix** — visual breakdown of TP/TN/FP/FN

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)

print('=== Evaluation Metrics ===')
print(f'  Accuracy  : {acc:.4f}  ({int(acc*len(y_test))}/{len(y_test)} correct predictions)')
print(f'  Precision : {prec:.4f}  (of students predicted Pass, {prec*100:.0f}% actually passed)')
print(f'  Recall    : {rec:.4f}  (model caught {rec*100:.0f}% of all actual passers)')
print(f'  F1 Score  : {f1:.4f}  (balanced precision-recall measure)')

print('\n=== Classification Report (per-class breakdown) ===')
print(classification_report(y_test, y_pred, target_names=['Fail', 'Pass']))

# Confusion matrix breakdown
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print('=== Confusion Matrix Breakdown ===')
print(f'  True  Negatives (TN) = {tn}  → correctly said FAIL')
print(f'  False Positives (FP) = {fp}  → said PASS but actually FAIL')
print(f'  False Negatives (FN) = {fn}  → said FAIL but actually PASS')
print(f'  True  Positives (TP) = {tp}  → correctly said PASS')

# Note: small test set (4 samples) = metrics look extreme (0 or 1)
# This is expected for a 20-row dataset — use more data in practice
print(f'\nNote: test set has only {len(y_test)} samples (20-row dataset is very small)')
print('In real projects, use at least 200+ rows for reliable evaluation.')

## Phase 7 — Visualisation

Two key plots:
1. **Confusion Matrix Heatmap** — visual grid of correct vs incorrect predictions
2. **Feature Influence** — which features push the model toward Pass or Fail?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Student Success Predictor — Results', fontsize=13, fontweight='bold')

# ── Plot 1: Confusion Matrix Heatmap ─────────────────────────────────────────
# Each cell shows count of predictions
# Diagonal (top-left, bottom-right) = correct predictions
# Off-diagonal = errors
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Predicted Fail', 'Predicted Pass'],
    yticklabels=['Actual Fail', 'Actual Pass'],
    ax=axes[0],
    linewidths=1,
    linecolor='white'
)
axes[0].set_title(f'Confusion Matrix\nAccuracy={acc:.2f}  F1={f1:.2f}')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ── Plot 2: Feature Coefficients ──────────────────────────────────────────────
# After scaling, magnitude of coefficient = relative importance
# Blue bars = feature pushes toward Pass
# Red bars  = feature pushes toward Fail
colors = ['mediumseagreen' if c > 0 else 'tomato' for c in coef_df['Coefficient']]
bars = axes[1].barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='black')
axes[1].axvline(0, color='black', lw=1.2)
axes[1].set_title('Feature Coefficients\n(green=towards Pass, red=towards Fail)')
axes[1].set_xlabel('Coefficient (after scaling)')
axes[1].grid(True, alpha=0.3, axis='x')

# Add value labels to bars
for bar, val in zip(bars, coef_df['Coefficient']):
    axes[1].text(
        val + (0.02 if val >= 0 else -0.02),
        bar.get_y() + bar.get_height() / 2,
        f'{val:+.3f}',
        va='center',
        ha='left' if val >= 0 else 'right',
        fontsize=9
    )

plt.tight_layout()
plt.show()

## Phase 8 — Predict a New Student

Given a new student's details, the model predicts whether they will Pass or Fail.

**Important**: always scale the new input using the **same scaler** fitted on training data.

In [ ]:
def predict_student(study_hours, attendance, past_score, sleep_hours):
    """
    Predict pass/fail for a new student.
    Uses the scaler and model trained above.
    """
    # Create input as a DataFrame with the same column names used in training
    new_data = pd.DataFrame([{
        'StudyHours': study_hours,
        'Attendance': attendance,
        'PastScore':  past_score,
        'SleepHours': sleep_hours,
    }])

    # Apply the SAME scaler from training — do NOT re-fit
    new_scaled = scaler.transform(new_data)

    # Predict class and probability
    prediction = model.predict(new_scaled)[0]          # 0 or 1
    probability = model.predict_proba(new_scaled)[0]   # [P(Fail), P(Pass)]

    result = 'PASS ✓' if prediction == 1 else 'FAIL ✗'
    return result, probability[1], probability[0]


# Test with a range of student profiles
test_students = [
    {'name': 'Hard Worker',    'study': 8, 'attend': 90, 'past': 80, 'sleep': 7},
    {'name': 'Average Student','study': 5, 'attend': 75, 'past': 60, 'sleep': 6},
    {'name': 'Struggling',     'study': 2, 'attend': 55, 'past': 40, 'sleep': 5},
    {'name': 'Sleepy Genius',  'study': 7, 'attend': 85, 'past': 75, 'sleep': 9},
    {'name': 'Borderline',     'study': 4, 'attend': 65, 'past': 55, 'sleep': 6},
]

print('=== Predictions for New Students ===')
print(f'{"Name":<18} {"Study":<7} {"Attend":<8} {"Past":<6} {"Sleep":<7} {"Result":<10} {"P(Pass)"}') 
print('-' * 70)
for s in test_students:
    result, p_pass, p_fail = predict_student(s['study'], s['attend'], s['past'], s['sleep'])
    print(f"{s['name']:<18} {s['study']:<7} {s['attend']:<8} {s['past']:<6} {s['sleep']:<7} {result:<10} {p_pass:.1%}")

print('\n=== Project Complete ===')
print('Pipeline summary:')
print('  1. Load CSV → EDA')
print('  2. Check missing values & outliers')
print('  3. LabelEncoder → convert Yes/No to 0/1')
print('  4. train_test_split → StandardScaler (fit on train only)')
print('  5. LogisticRegression.fit(X_train, y_train)')
print('  6. Evaluate: accuracy, precision, recall, F1, confusion matrix')
print('  7. Visualise: heatmap + coefficient plot')
print('  8. Predict: scaler.transform(new_data) → model.predict()')